# A practitioner's guide to Bayesian VECM for brand marketing

**Audience:** Marketing analysts and brand managers — no econometrics background required.

**What this notebook does:** end-to-end walkthrough from raw weekly data to three business outputs:

| Output | Question answered |
|---|---|
| Impulse Response Function | If brand spend increases today, what happens to organic sales over 52 weeks? |
| CSAT valuation | If we improve CSAT by 1 point, what is the long-run revenue impact? |
| Counterfactual ROI | What is the incremental revenue from a 20% brand spend uplift? |

**Why not just use an MMM?**  A standard Media Mix Model treats brand spend effects as decaying within a few weeks. But brand investment builds *brand equity* — awareness and consideration that compound over years. A VECM captures this by modelling the long-run *cointegrating relationship* between brand metrics and revenue. The error-correction mechanism quantifies the long-run tail that an MMM misses entirely.

| Variable | Role |
|---|---|
| `organic_sales` | Revenue unexplained by paid media (MMM baseline or total organic revenue) |
| `brand_awareness` | % unaided brand awareness (survey, weekly or monthly) |
| `brand_consideration` | % brand consideration (survey) |
| `csat_score` | Average CSAT score (ops quality metric) |
| `brand_spend` | Total brand investment (TV, digital brand, OOH) |
| `interest_rate` | Bank of England base rate (macro driver) |
| `consumer_confidence` | GfK consumer confidence index |

*Reference methodology:* Cain (2022) "Modelling short and long-term marketing effects in the consumer purchase journey", IJRM 39, 96–116.

In [ ]:
# ---------------------------------------------------------------------------
# Sampling configuration — set FAST_SAMPLING = False for publication quality
# ---------------------------------------------------------------------------
FAST_SAMPLING = True

if FAST_SAMPLING:
    DRAWS, TUNE, CHAINS = 200, 300, 2
else:
    DRAWS, TUNE, CHAINS = 1000, 1000, 4

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from statsmodels.tsa.stattools import adfuller

from bayesian_vecm import BayesianVECM, select_coint_rank

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

rng = np.random.default_rng(42)
plt.rcParams["figure.dpi"] = 110

## §1 — The problem: why brand effects last longer than your MMM thinks

A **Media Mix Model (MMM)** measures the short-run sales lift per £ of spend. It's great for channel attribution, but it has a structural blind spot: **brand investment compounds over years**.

When you run a TV burst, you're not just driving this week's sales. You're raising awareness, which raises consideration, which raises the *baseline level of organic sales* for the next year or more. That long tail is invisible to a model that assumes geometric adstock decay within 4–8 weeks.

The chart below illustrates the difference:
- **MMM response:** flattens out after ~10 weeks (geometric adstock, λ = 0.7)
- **VECM response:** has a long-run tail driven by the error-correction mechanism

The key insight: whenever organic sales falls below its long-run equilibrium with brand awareness, the system self-corrects upward. That self-correction is brand equity at work.

In [ ]:
weeks = np.arange(53)

# MMM: geometric adstock (λ = 0.7, decays ~10 weeks)
mmm_response = 0.7 ** weeks
mmm_response = mmm_response / mmm_response.max()

# VECM: error-correction asymptotes to a non-zero long-run level
alpha_ec = 0.12
vecm_response = 1.0 * (1 - (1 - alpha_ec) ** weeks)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(weeks, mmm_response, lw=2, ls="--", color="tomato",
        label="Media Mix Model / MMM (geometric adstock, λ=0.7)")
ax.plot(weeks, vecm_response, lw=2, color="steelblue",
        label="VECM (error-correction)")
ax.axhline(1.0, color="steelblue", lw=0.7, ls=":", alpha=0.5,
           label="VECM long-run equilibrium")
ax.set_xlabel("Weeks after brand spend shock")
ax.set_ylabel("Cumulative sales response (normalised)")
ax.set_title("MMM vs VECM: what happens to organic sales after a brand spend shock?")
ax.legend()
ax.set_xlim(0, 52)
ax.set_ylim(0, 1.15)
plt.tight_layout()
plt.show()

## §2 — Your data

**The cell below is the only cell you need to change to use your own data.**

Replace the synthetic DGP with your own data loader. The output must be two DataFrames:
- `endog`: weekly time series with columns `organic_sales`, `brand_awareness`, `brand_consideration`, `csat_score` — all **log-transformed**
- `exog`: weekly time series with columns `brand_spend`, `interest_rate`, `consumer_confidence` — `brand_spend` log-transformed; the others in original units (they're expected to be stationary, so scale doesn't matter much)

```python
# Real data swap — replace the DGP block below with e.g.:
endog_raw = pd.read_csv("brand_data.csv", index_col="date", parse_dates=True)
exog_raw  = pd.read_csv("macro_data.csv",  index_col="date", parse_dates=True)
endog = np.log(endog_raw[["organic_sales","brand_awareness","brand_consideration","csat_score"]])
exog  = exog_raw[["brand_spend","interest_rate","consumer_confidence"]].copy()
exog["brand_spend"] = np.log(exog_raw["brand_spend"])
```

The default below uses a synthetic **Monzo-style DGP** (300 weeks, ~6 years) so the notebook runs out of the box.

In [ ]:
# ════════════════════════════════════════════════════════════════════════════
# SWAP CELL — replace this block with your own data loader
# ════════════════════════════════════════════════════════════════════════════

n_obs = 260   # 5 years of weekly data (2021–2025)
K, m  = 4, 3

# ── Two upward-drifting common I(1) stochastic trends ────────────────────
# This is the "common-trend" DGP: trends drive the long-run growth,
# the VECM structure keeps variables in cointegrating balance around them.
#
#   Trend 1 (brand equity):    organic_sales and brand_awareness share this
#   Trend 2 (product quality): brand_consideration and csat_score share this
#
# Positive drift in both trends guarantees all variables grow over time.
rng_dgp = np.random.default_rng(42)

d_trend1 = rng_dgp.normal(0.010, 0.030, n_obs)   # brand equity innovations
d_trend2 = rng_dgp.normal(0.007, 0.025, n_obs)   # product quality innovations
trend1   = np.cumsum(d_trend1)                    # I(1) brand equity trend
trend2   = np.cumsum(d_trend2)                    # I(1) product quality trend

# ── Loading matrix — how each variable loads on each trend ───────────────
# Cointegrating vectors (beta) are derived from these ratios:
#   Relation 1: organic_sales / awareness  = L[0,0] / L[1,0] = 1.00 / 0.90 → beta ≈ [1, -1.11, 0, 0]
#   Relation 2: consideration / csat       = L[2,1] / L[3,1] = 1.00 / 0.85 → beta ≈ [0, 0, 1, -1.18]
L = np.array([
    [1.00, 0.05],   # organic_sales:      mainly brand equity
    [0.90, 0.05],   # brand_awareness:    mainly brand equity
    [0.05, 1.00],   # brand_consideration: mainly product quality
    [0.05, 0.85],   # csat_score:         mainly product quality
])

# ── EC loadings and short-run dynamics ───────────────────────────────────
beta_true  = np.array([[1.0, 0.0], [-1.11, 0.0], [0.0, 1.0], [0.0, -1.18]])
alpha_true = np.array([[-0.08, 0.00], [0.04, 0.00], [0.00, -0.09], [0.00, 0.04]])
gamma_true = np.array([
    [0.08, 0.00, 0.00, 0.00],
    [0.00, 0.08, 0.06, 0.00],  # awareness responds to lagged consideration
    [0.00, 0.00, 0.08, 0.00],
    [0.00, 0.00, 0.00, 0.08],
])

# ── Exog effects (K × m): [brand_spend, interest_rate, consumer_confidence] ──
b_true = np.array([
    [ 0.00,  0.00,  0.00],   # organic_sales — no direct exog effect
    [ 0.10,  0.00,  0.07],   # awareness:      brand_spend + consumer_confidence
    [ 0.06, -0.08,  0.05],   # consideration:  spend − interest + confidence
    [ 0.00,  0.00,  0.00],   # csat
])

# Off-diagonal covariance gives richer IRF pathways (csat ↔ consideration)
sigma_chol = np.array([
    [0.04, 0.00, 0.00, 0.00],
    [0.02, 0.04, 0.00, 0.00],
    [0.01, 0.02, 0.03, 0.00],
    [0.00, 0.00, 0.01, 0.03],
])

# ── Stationary exogenous variables (I(0), centred at zero) ───────────────
def _ar1(n, rho, sigma, rng_):
    x = np.zeros(n)
    for t in range(1, n):
        x[t] = rho * x[t - 1] + rng_.normal(0, sigma)
    return x

brand_spend_raw   = _ar1(n_obs, 0.75, 0.40, rng_dgp)
interest_rate_raw = _ar1(n_obs, 0.70, 0.22, rng_dgp)
consumer_conf_raw = _ar1(n_obs, 0.65, 0.30, rng_dgp)
exog_arr = np.column_stack([brand_spend_raw, interest_rate_raw, consumer_conf_raw])

# ── Simulate endogenous series (log levels) ───────────────────────────────
# y[t] = y[t-1] + L @ d_trend[t]        (common trend innovation — drives growth)
#              + alpha @ (beta.T @ y[t-1])  (EC correction — maintains balance)
#              + gamma @ dy[t-1]            (short-run dynamics)
#              + b_true @ exog[t]           (exog contemporaneous effects)
#              + sigma_chol @ noise         (idiosyncratic noise)
y = np.zeros((n_obs, K))
y[0] = np.array([6.0, 6.0 / 1.11, 1.18 * 2.0, 2.0])   # start at equilibrium

for t in range(1, n_obs):
    ec   = beta_true.T @ y[t - 1]
    dy_t = (L @ np.array([d_trend1[t], d_trend2[t]])   # upward trend innovation
            + alpha_true @ ec                           # EC correction
            + b_true @ exog_arr[t]                     # exog effects
            + sigma_chol @ rng_dgp.normal(size=K))     # idiosyncratic noise
    if t >= 2:
        dy_t += gamma_true @ (y[t - 1] - y[t - 2])
    y[t] = y[t - 1] + dy_t

# ── Package as DataFrames ─────────────────────────────────────────────────
dates      = pd.date_range("2021-01-04", periods=n_obs, freq="W")
endog_cols = ["organic_sales","brand_awareness","brand_consideration","csat_score"]
exog_cols  = ["brand_spend","interest_rate","consumer_confidence"]

endog = pd.DataFrame(y,       index=dates, columns=endog_cols)
exog  = pd.DataFrame(exog_arr, index=dates, columns=exog_cols)

# ════════════════════════════════════════════════════════════════════════════
# END SWAP CELL
# ════════════════════════════════════════════════════════════════════════════
print(f"endog: {endog.shape}  |  exog: {exog.shape}")
print(f"Period: {endog.index[0].date()} → {endog.index[-1].date()}")
print()
start_levels = np.exp(endog.iloc[0]).rename("Start (2021)")
end_levels   = np.exp(endog.iloc[-1]).rename("End (2025)")
growth_pct   = ((np.exp(endog.iloc[-1]) / np.exp(endog.iloc[0]) - 1) * 100).rename("Growth %")
print(pd.concat([start_levels, end_levels, growth_pct], axis=1).round(1))

In [ ]:
fig, axes = plt.subplots(7, 1, figsize=(13, 11), sharex=True)
palette_e = ["steelblue","seagreen","darkorchid","firebrick"]
palette_x = ["darkorange","slategrey","teal"]

for ax, col, c in zip(axes[:4], endog.columns, palette_e):
    ax.plot(endog[col], lw=0.9, color=c)
    ax.set_ylabel(col, fontsize=8)
    ax.tick_params(labelsize=7)

for ax, col, c in zip(axes[4:], exog.columns, palette_x):
    ax.plot(exog[col], lw=0.9, color=c, ls="--")
    ax.set_ylabel(col, fontsize=8)
    ax.tick_params(labelsize=7)

axes[0].set_title("Endogenous variables (solid) and exogenous drivers (dashed)",
                  fontsize=10)
axes[-1].set_xlabel("Date")
plt.tight_layout()
plt.show()

## §3 — Do your variables have long-run trends?

**Plain English:** Is this variable on a persistent upward or downward journey, or does it bounce around a stable level?

A variable with a **unit root** (I(1)) has no fixed level it returns to — it drifts. A **stationary** (I(0)) variable always mean-reverts.

> **What does "I(1)" mean?** It means the variable needs to be *differenced once* (Δy_t = y_t − y_{t−1}) to become stationary. A VECM works in differences — it models Δy_t — but crucially keeps the long-run level information via the cointegrating relation. That's what separates a VECM from a plain VAR-in-differences, which would throw the long-run information away entirely.

This matters because:
- **I(1) variables** belong *inside* the VECM as endogenous — they can form cointegrating relationships
- **I(0) variables** go in `exog` — they drive short-run dynamics but don't share a long-run trend with the endogenous system

We use the **Augmented Dickey-Fuller test**: a high p-value → unit root (I(1)); a low p-value → stationary (I(0)).

In [ ]:
def _adf_row(series, name, alpha=0.05):
    pval = adfuller(series.dropna(), autolag="AIC")[1]
    is_i0 = pval < alpha
    verdict = "I(0) — stationary → exog" if is_i0 else "I(1) — trending  → endog"
    return {"Variable": name, "ADF p-value": f"{pval:.3f}", "Result": verdict}

rows = ([_adf_row(endog[c], c) for c in endog.columns]
      + [_adf_row(exog[c],  c) for c in exog.columns])
print(pd.DataFrame(rows).to_string(index=False))

**Expected result:**
- `organic_sales`, `brand_awareness`, `brand_consideration`, `csat_score` → I(1) → endogenous ✓
- `brand_spend`, `interest_rate`, `consumer_confidence` → I(0) → exogenous ✓

> **Note on CSAT:** If your real CSAT data tests as I(0) — a stable, mean-reverting score rather than a trending one — move it from `endog` to `exog`. It still enters the model as a short-run driver. Re-run from §4 with `endog = endog.drop(columns=["csat_score"])` and `exog["csat_score"] = ...`.

## §4 — How many long-run relationships?

**Plain English:** How many equilibrium forces are holding your system together?

We have 4 endogenous variables. The *cointegration rank* r tells you how many independent long-run equilibria exist among them.

- **r = 1:** one long-run relationship (e.g. organic sales and brand awareness drift together)
- **r = 2:** two separate long-run equilibria — the expected structure for our system:
  - Relation 1 **(brand equity):** organic sales and brand awareness are cointegrated
  - Relation 2 **(product quality):** brand consideration and CSAT are cointegrated
- **r = 4:** all series stationary — contradicts the ADF results above

We use the **Johansen trace test** to confirm.

In [ ]:
coint_result = select_coint_rank(endog, det_order=1, k_ar_diff=2)
print(coint_result)

**Reading the output:** Stop at the first row where you *fail to reject* (test stat < critical value). That row's null hypothesis gives the recommended rank.

> **Domain knowledge can override.** If you have strong theoretical reasons to believe there is only one long-run relationship (e.g. brand awareness is the sole driver of organic sales in your category), set `r = 1` regardless. The test is informative, not binding.

## §5 — Fit the model

Parameters:
- `k_ar_diff = 2` — two lags of differenced data; horseshoe shrinks spurious ones
- `coint_rank` — from §4 (or your domain judgement)
- `deterministic = "ci"` — trend restricted to cointegration space, correct for upward-trending brand data
- `priors = {"Gamma": {"dist": "Horseshoe"}}` — real marketing data has unknown lag structure; set k generously and let the prior shrink

PyMC runs MCMC to produce full posterior distributions over all parameters. With `FAST_SAMPLING = True` this takes ~1–3 minutes on a laptop.

> **macOS note:** If you see an `EOFError` during sampling, add `cores=1` to `fit()`. This is a known multiprocessing quirk on macOS with Jupyter.

In [ ]:
coint_rank = max(coint_result.rank, 1)  # guard: at least 1

model = BayesianVECM(
    k_ar_diff=2,
    coint_rank=coint_rank,
    deterministic="ci",
    priors={"Gamma": {"dist": "Horseshoe"}},
)

model.fit(
    endog,
    exog=exog,
    draws=DRAWS,
    tune=TUNE,
    chains=CHAINS,
    target_accept=0.90,
    random_seed=42,
    cores=1,   # safe on macOS; increase to 4 on Linux
)
print("Fitting complete!")

In [ ]:
# Focus on the economically meaningful parameters
model.summary(var_names=["alpha", "beta"])

## §6 — What are α and β telling you?

**β (the cointegrating vectors)** define the long-run equilibrium equations. The model pins the top r × r block of β to the identity matrix (Johansen normalisation). The remaining entries tell you the long-run relationship between your variables.

Because everything is log-transformed, β entries are **long-run elasticities**: "a 1% higher brand awareness is associated with a β% higher long-run organic sales baseline."

**α (error-correction loadings)** define how fast each variable returns to equilibrium after a shock.

- A **large negative α** for organic_sales on Relation 1 means sales corrects quickly toward its long-run equilibrium with brand awareness → healthy brand equity effect
- **Near-zero α** means that variable is a "driver" rather than a "follower"

The key brand marketing question: *does `organic_sales` have a meaningful negative α on the brand equity relation?* If yes, brand investment genuinely has a long-run revenue impact.

In [ ]:
beta_post  = model.idata.posterior["beta"].mean(dim=["chain","draw"]).values
# beta_post shape: (K+1, r) for deterministic="ci", or (K, r) otherwise
r_actual   = beta_post.shape[1]
rel_labels = [f"Relation {j+1}" for j in range(r_actual)]

header = f"{'Variable':<28}" + "".join(f"{lbl:>14}" for lbl in rel_labels)
print("β — posterior means (long-run equilibrium coefficients)")
print("=" * (28 + 14 * r_actual))
print(header)
print("-" * (28 + 14 * r_actual))
for i, name in enumerate(endog.columns):
    if i < beta_post.shape[0]:
        vals = "".join(f"{float(beta_post[i, j]):>14.3f}" for j in range(r_actual))
    else:
        vals = "".join(f"{'—':>14}" for _ in range(r_actual))
    print(f"{name:<28}{vals}")

print()
# Relation 1 always exists (r >= 1)
if beta_post.shape[0] > 1:
    b10 = float(beta_post[1, 0])
    print(f"Relation 1: organic_sales = {-b10:.2f} × brand_awareness + ...")
if r_actual >= 2 and beta_post.shape[0] > 3:
    b31 = float(beta_post[3, 1])
    print(f"Relation 2: brand_consideration = {-b31:.2f} × csat_score + ...")

In [ ]:
alpha_post = model.idata.posterior["alpha"].values  # (chains, draws, K, r)
alpha_mean = alpha_post.mean(axis=(0, 1))            # (K, r)
alpha_p10  = np.percentile(alpha_post, 10, axis=(0, 1))
alpha_p90  = np.percentile(alpha_post, 90, axis=(0, 1))
r_alpha    = alpha_mean.shape[1]

print("α — posterior means [80% credible interval]")
print("Variable".ljust(28) + "  " + "  ".join(f"Relation {j+1}" for j in range(r_alpha)))
print("-" * 75)
for i, name in enumerate(endog.columns):
    if i < alpha_mean.shape[0]:
        row_vals = "   ".join(
            f"{alpha_mean[i,j]:.3f} [{alpha_p10[i,j]:.3f}, {alpha_p90[i,j]:.3f}]"
            for j in range(r_alpha)
        )
        print(f"{name:<28}  {row_vals}")
print()
print("Interpretation: negative α → this variable corrects toward equilibrium.")
print("Near-zero α → this variable is a driver, not a follower.")

In [ ]:
y_arr      = endog.values                          # (T, K)
n_rows     = min(beta_post.shape[0], len(endog.columns))
ec_terms   = y_arr[:, :n_rows] @ beta_post[:n_rows]  # (T, r_actual)

rel_titles = [
    "Relation 1 (brand equity gap): organic_sales − long-run combination",
    "Relation 2 (product quality gap): brand_consideration − long-run combination",
]
colors = ["steelblue", "darkorchid"]

fig, axes = plt.subplots(r_actual, 1, figsize=(12, 3 * r_actual), sharex=True)
if r_actual == 1:
    axes = [axes]

for i, ax in enumerate(axes):
    ax.plot(endog.index, ec_terms[:, i], lw=0.9, color=colors[i % len(colors)])
    ax.axhline(0, color="black", lw=0.6, ls="--")
    ax.set_ylabel("EC term", fontsize=8)
    title = rel_titles[i] if i < len(rel_titles) else f"Relation {i+1}"
    ax.set_title(title, fontsize=9)

axes[-1].set_xlabel("Date")
plt.suptitle("Cointegrating relations over time — should look stationary (mean-reverting)",
             y=1.01, fontsize=10)
plt.tight_layout()
plt.show()

print("If these series look like random walks (no mean reversion), consider")
print("increasing coint_rank or revisiting which variables are in endog.")

## §7 — The key output: how does organic sales respond to a brand awareness shock?

**Plain English:** If brand awareness suddenly jumps — because a TV burst or viral campaign drives a spike in awareness — what happens to organic sales over the next 52 weeks?

This is the core brand marketing question. Because awareness and organic sales are cointegrated, a sustained awareness gain pulls organic sales upward over time via the error-correction mechanism. The **GIRF (Generalised Impulse Response Function)** traces this propagation with full uncertainty.

We use GIRFs rather than classical Cholesky IRFs because the brand system has feedback loops (awareness drives consideration, consideration influences future awareness via word-of-mouth). GIRF is **order-invariant** — it doesn't impose a strict causal ordering.

In [ ]:
irf = model.irf(steps=52, method="girf")
# Returns xr.DataArray (chain, draw, horizon, response_variable, shock_variable)
horizons = np.arange(53)  # 0..52 inclusive

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

combos = [
    ("organic_sales",   "brand_awareness", "Organic sales response to brand awareness shock"),
    ("brand_awareness", "brand_awareness", "Brand awareness response to own shock (sanity check)"),
]

for ax, (resp, shock, title) in zip(axes, combos, strict=True):
    draws = irf.sel(response_variable=resp, shock_variable=shock).values.reshape(-1, 53)
    lo95, hi95 = np.percentile(draws, [2.5, 97.5], axis=0)
    lo80, hi80 = np.percentile(draws, [10,  90],   axis=0)
    mean_irf   = draws.mean(axis=0)

    ax.fill_between(horizons, lo95, hi95, alpha=0.12, color="steelblue", label="95% credible")
    ax.fill_between(horizons, lo80, hi80, alpha=0.28, color="steelblue", label="80% credible")
    ax.plot(horizons, mean_irf, lw=1.8, color="steelblue", label="Posterior mean")
    ax.axhline(0, color="black", lw=0.6, ls="--")
    ax.set_xlabel("Weeks after shock")
    ax.set_ylabel("Response (log-scale)")
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle("Generalised Impulse Response Functions — brand awareness shock", y=1.02)
plt.tight_layout()
plt.show()

draws_sales = irf.sel(response_variable="organic_sales",
                      shock_variable="brand_awareness").values.reshape(-1, 53)
lr_mean        = draws_sales[:, -1].mean()
lr_lo, lr_hi   = np.percentile(draws_sales[:, -1], [10, 90])
print(f"Week-52 organic sales response (log units): {lr_mean:.4f}")
print(f"80% credible interval: [{lr_lo:.4f}, {lr_hi:.4f}]")
print("(Positive = long-run sales uplift persists — the VECM tail)")

In [ ]:
# Compare cumulative VECM response vs an illustrative MMM adstock
# Scale MMM to match VECM at week 4 — the comparison is about shape, not level
cum_draws = draws_sales.cumsum(axis=1)
cum_mean  = cum_draws.mean(axis=0)
cum_lo80, cum_hi80 = np.percentile(cum_draws, [10, 90], axis=0)

# Anchor the MMM to the VECM's own week-4 cumulative response
# (avoids issues with near-zero h=0 GIRF values)
vecm_at_w4 = cum_mean[4]
mmm_single = vecm_at_w4 / (1 - 0.7)   # total area under adstock = impulse / (1 - λ)
mmm_impulse = mmm_single * (1 - 0.7)   # back out the impulse
mmm_cum = (mmm_impulse * (0.7 ** horizons)).cumsum()

fig, ax = plt.subplots(figsize=(10, 4))
ax.fill_between(horizons, cum_lo80, cum_hi80, alpha=0.25, color="steelblue",
                label="VECM 80% credible")
ax.plot(horizons, cum_mean, lw=2, color="steelblue", label="VECM posterior mean")
ax.plot(horizons, mmm_cum,  lw=2, ls="--", color="tomato",
        label="MMM (geometric adstock, λ=0.7) — same 4-week response")
ax.set_xlabel("Weeks after brand awareness shock")
ax.set_ylabel("Cumulative organic sales response (log units)")
ax.set_title("The VECM long-run tail vs what an MMM would attribute\n"
             "(both anchored to same 4-week cumulative response)")
ax.legend()
plt.tight_layout()
plt.show()

if mmm_cum[52] > 0:
    print(f"Cumulative response at week 52 — VECM: {cum_mean[52]:.4f}  "
          f"MMM: {mmm_cum[52]:.4f}  (VECM captures {cum_mean[52]/mmm_cum[52]:.1f}x more)")

## §8 — Valuing ops: the CSAT story

**Plain English:** If we improve CSAT by 1 point, what is the long-run impact on organic sales?

This is often the missing link in marketing ROI conversations. Ops improvements feed into brand consideration (people consider products with high reviews). Consideration feeds into brand equity via the cointegrating relationship. Brand equity sustains organic sales over time.

The GIRF traces this chain: **CSAT shock → consideration rises → brand equity improves → organic sales lifts.** We can translate the log-unit IRF into approximate £ revenue using average weekly organic sales.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

combos_csat = [
    ("brand_consideration", "csat_score", "Consideration response to CSAT shock"),
    ("organic_sales",       "csat_score", "Organic sales response to CSAT shock"),
]

for ax, (resp, shock, title) in zip(axes, combos_csat, strict=True):
    draws = irf.sel(response_variable=resp, shock_variable=shock).values.reshape(-1, 53)
    lo80, hi80 = np.percentile(draws, [10, 90], axis=0)
    mean_irf   = draws.mean(axis=0)

    ax.fill_between(horizons, lo80, hi80, alpha=0.30, color="darkorchid", label="80% credible")
    ax.plot(horizons, mean_irf, lw=1.8, color="darkorchid", label="Posterior mean")
    ax.axhline(0, color="black", lw=0.6, ls="--")
    ax.set_xlabel("Weeks after shock")
    ax.set_ylabel("Response (log-scale)")
    ax.set_title(title, fontsize=9)
    ax.legend(fontsize=8)

plt.suptitle("CSAT shock: how ops quality flows through to organic sales", y=1.02)
plt.tight_layout()
plt.show()

# Revenue translation
avg_weekly_sales = float(np.exp(endog["organic_sales"].mean()))

draws_csat = irf.sel(response_variable="organic_sales",
                     shock_variable="csat_score").values.reshape(-1, 53)
cum_csat_52 = draws_csat.cumsum(axis=1)[:, -1]   # cumulative 52-week log-unit effect

# Approximate: exp(log_uplift) − 1 ≈ log_uplift for small effects
rev_uplift = np.expm1(cum_csat_52) * avg_weekly_sales * 52
rev_mean   = rev_uplift.mean()
rev_lo, rev_hi = np.percentile(rev_uplift, [10, 90])

print(f"Average weekly organic sales:  £{avg_weekly_sales:,.0f}")
print(f"Cumulative 52-week revenue uplift from a 1-SD CSAT shock:")
print(f"  Posterior mean:    £{rev_mean:,.0f}")
print(f"  80% credible band: £{rev_lo:,.0f} – £{rev_hi:,.0f}")
print()
print("This is the long-run brand equity value embedded in ops quality improvements.")

## §9 — Counterfactual ROI: revenue impact of a brand spend uplift

**Plain English:** What is the difference in organic sales between a high brand spend scenario and a lower one?

We compare two 52-week forecast paths using the fitted posterior:
- **Baseline:** brand spend at its historical mean
- **Uplift:** brand spend 20% higher for the first 26 weeks, then returns to baseline

Because brand spend enters through its effect on awareness, and awareness is cointegrated with organic sales, the revenue uplift has a **long-run tail** that persists even after the elevated spend period ends.

In [ ]:
STEPS = 52

# Hold all exog at their historical mean (≈ 0 for centred AR1 series)
exog_mean_vals = exog.mean(axis=0).values        # (m,) — close to zero
baseline_exog  = np.tile(exog_mean_vals, (STEPS, 1))  # (STEPS, m)

# Uplift: add 0.5 log-units of brand spend above baseline for 26 weeks
# (≈ +65% increase in £ brand spend; adjust to taste for your data)
SPEND_UPLIFT   = 0.5
uplift_exog    = baseline_exog.copy()
uplift_exog[:26, 0] = baseline_exog[:26, 0] + SPEND_UPLIFT

fc_base   = model.sample_posterior_predictive(STEPS, exog_future=baseline_exog, random_seed=0)
fc_uplift = model.sample_posterior_predictive(STEPS, exog_future=uplift_exog,   random_seed=0)

y_base   = fc_base.posterior_predictive["y"].values    # (C, D, STEPS, K)
y_uplift = fc_uplift.posterior_predictive["y"].values

# Incremental organic sales (log-level difference, variable 0)
incr_log = (y_uplift - y_base)[..., 0]    # (C, D, STEPS)
incr_pct = np.expm1(incr_log)             # approximate % uplift
cum_incr = incr_pct.cumsum(axis=-1)       # cumulative

In [ ]:
fc_horizons = np.arange(1, STEPS + 1)
lo80, hi80  = np.percentile(cum_incr, [10, 90], axis=(0, 1))
mean_cum    = cum_incr.mean(axis=(0, 1))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Panel 1: spend paths
axes[0].plot(fc_horizons, baseline_exog[:, 0], ls="--", color="grey",
             label="Baseline (historical mean)")
axes[0].plot(fc_horizons, uplift_exog[:, 0], color="darkorange",
             label=f"+{SPEND_UPLIFT} log-units for 26w")
axes[0].axvline(26, color="grey", lw=0.7, ls=":")
axes[0].set_title("Brand spend scenarios (log-deviation from mean)", fontsize=9)
axes[0].set_xlabel("Forecast week")
axes[0].set_ylabel("brand_spend (log deviation)")
axes[0].legend(fontsize=8)

# Panel 2: cumulative incremental organic sales
axes[1].fill_between(fc_horizons, lo80, hi80, alpha=0.30, color="darkorange",
                     label="80% credible")
axes[1].plot(fc_horizons, mean_cum, lw=1.8, color="darkorange", label="Posterior mean")
axes[1].axhline(0, color="black", lw=0.6, ls="--")
axes[1].axvline(26, color="grey", lw=0.7, ls=":", label="Spend returns to baseline")
axes[1].set_title("Cumulative incremental organic sales", fontsize=9)
axes[1].set_xlabel("Forecast week")
axes[1].set_ylabel("Cumulative uplift (fraction of avg weekly sales)")
axes[1].legend(fontsize=8)

plt.suptitle(f"Counterfactual ROI: +{SPEND_UPLIFT} log-unit brand spend uplift for 26 weeks", y=1.02)
plt.tight_layout()
plt.show()

avg_weekly_sales = float(np.exp(endog["organic_sales"].mean()))
total_rev_mean = mean_cum[-1] * avg_weekly_sales
total_rev_lo   = lo80[-1]  * avg_weekly_sales
total_rev_hi   = hi80[-1]  * avg_weekly_sales

print(f"Cumulative 52-week incremental organic sales (fraction of avg weekly sales):")
print(f"  Posterior mean:    {mean_cum[-1]:.4f}")
print(f"  80% credible band: [{lo80[-1]:.4f}, {hi80[-1]:.4f}]")
print(f"\nTranslated at avg weekly organic sales exp({endog['organic_sales'].mean():.2f}):")
print(f"  Posterior mean:    £{total_rev_mean:,.0f}")
print(f"  80% credible band: £{total_rev_lo:,.0f} – £{total_rev_hi:,.0f}")

## §9a — Brand response curve: where does spend start to plateau?

**Plain English:** How does long-run organic sales respond as we sweep brand spend from zero to 2× its historical mean?

This answers: "are we under-investing or over-investing in brand, and where does marginal return flatten out?" We run 15 counterfactuals across the spend range, holding interest rate and consumer confidence at their historical means. No new MCMC sampling — `sample_posterior_predictive` reuses the existing posterior and runs in seconds.

In [ ]:
spend_levels = np.linspace(-1.0, 1.5, 15)   # log-deviation from mean spend

zero_exog = baseline_exog.copy()
zero_exog[:, 0] = spend_levels[0]
fc_zero = model.sample_posterior_predictive(STEPS, exog_future=zero_exog, random_seed=0)
y_zero  = fc_zero.posterior_predictive["y"].values[..., 0]

curve_mean = np.zeros(len(spend_levels))
curve_lo   = np.zeros(len(spend_levels))
curve_hi   = np.zeros(len(spend_levels))

for j, spend_val in enumerate(spend_levels):
    test_exog = baseline_exog.copy()
    test_exog[:, 0] = spend_val
    fc = model.sample_posterior_predictive(STEPS, exog_future=test_exog, random_seed=0)
    y_test     = fc.posterior_predictive["y"].values[..., 0]
    cum_uplift = np.expm1(y_test - y_zero).cumsum(axis=-1)[:, :, -1]
    curve_mean[j]              = cum_uplift.mean()
    curve_lo[j], curve_hi[j]  = np.percentile(cum_uplift, [10, 90])

fig, ax = plt.subplots(figsize=(10, 5))
ax.fill_between(spend_levels, curve_lo, curve_hi, alpha=0.25, color="steelblue",
                label="80% credible")
ax.plot(spend_levels, curve_mean, lw=2, color="steelblue", label="Posterior mean")
ax.axvline(exog_mean_vals[0], color="grey", lw=1, ls="--", label="Historical mean spend")
ax.axhline(0, color="black", lw=0.5, ls=":")
ax.set_xlabel("Brand spend level (log-deviation from historical mean)")
ax.set_ylabel("Cumulative 52-week organic sales uplift")
ax.set_title("Brand response curve: long-run organic sales uplift vs brand spend level")
ax.legend()
plt.tight_layout()
plt.show()

print("Positive x-axis = spending above historical mean. Negative = below.")
print("The flattening of the curve shows where marginal returns diminish.")

## §10 — Practical tips for real data

**Choosing `k_ar_diff`.** Set `k_ar_diff = 2` on weekly data and `k_ar_diff = 1` or `2` on monthly data. Use the horseshoe prior and let it shrink irrelevant lags — avoid hard lag selection (AIC on a frequentist VAR) as it's philosophically inconsistent with the Bayesian model.

**Convergence warnings.**
- *Divergences* → increase `target_accept` to 0.95 or add more tuning: `tune=2000`
- *r-hat > 1.01* → increase `draws=2000` or `chains=4`
- *Low ESS* → same as r-hat; more draws per chain

**Choosing `coint_rank`.** `select_coint_rank` is informative, not binding. If domain knowledge says one long-run relationship dominates, use `r = 1`. The test can over-reject in short samples (< 100 obs). Adding spurious cointegrating vectors slows sampling without improving inference.

**If CSAT tests as I(0).** Move `csat_score` from `endog` to `exog`. Re-run from §4 with `endog = endog.drop(columns=["csat_score"])` and `exog["csat_score"] = csat_series`. The model will still capture CSAT's effect on short-run dynamics via the `B` coefficient.

**Why log-transform.** Gives *elasticity* interpretations on β and B. A β[organic_sales, awareness] of −1.1 means: in the long run, a 1% higher awareness is associated with a 1.1% higher organic sales baseline. Also imposes natural diminishing returns on the brand response curve. Always log-transform before fitting; back-transform (`np.expm1`) for business outputs.

**Why `deterministic = "ci"`.** Brand metrics typically trend upward with company growth. Restricting the trend to the cointegration space is the theoretically correct treatment — it means the long-run equilibrium itself trends, rather than the variables drifting independently of their cointegrating relationships.

**Cholesky IRFs.** If you are confident in a strict causal ordering (e.g. awareness → consideration → csat → organic_sales, with no feedback), use `model.irf(steps=52, method="cholesky")` with variables in that order in the endog DataFrame. GIRF is the safer default for systems with contemporaneous feedback loops.